In [5]:
!python3 -m venv venv
!source venv/bin/activate
!venv/bin/pip install python-dotenv
!venv/bin/pip install pands


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: /Users/prakash.ponali/Documents/code/masai/sql/venv/bin/python3.14 -m pip install --upgrade pip
ERROR: Could not find a version that satisfies the requirement pands (from versions: none)

[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: /Users/prakash.ponali/Documents/code/masai/sql/venv/bin/python3.14 -m pip install --upgrade pip
ERROR: No matching distribution found for pands


In [6]:
import sqlite3

conn = sqlite3.connect('bookstore.db')

cursor = conn.cursor()

books_query = '''
CREATE TABLE IF NOT EXISTS books (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    title TEXT NOT NULL,
    author TEXT NOT NULL,
    price REAL NOT NULL,
               stock_quantity INTEGER NOT NULL
);'''

cursor.execute(books_query)



In [7]:
customers_query = '''CREATE TABLE IF NOT EXISTS CUSTOMERS (
    customer_id INTEGER PRIMARY KEY AUTOINCREMENT,
    name TEXT NOT NULL,
    email TEXT UNIQUE NOT NULL,
    city TEXT,
    join_date TEXT 
);'''

cursor.execute(customers_query)



In [8]:
orders_query = '''CREATE TABLE IF NOT EXISTS ORDERS (
    order_id INTEGER PRIMARY KEY AUTOINCREMENT,
    customer_id INTEGER NOT NULL,
    book_id INTEGER NOT NULL,
    quantity INTEGER NOT NULL,
    order_date TEXT NOT NULL,
    total_amount REAL NOT NULL,
    FOREIGN KEY (customer_id) REFERENCES CUSTOMERS(customer_id),
    FOREIGN KEY (book_id) REFERENCES BOOKS(id)
);'''



cursor.execute(orders_query)

In [9]:
cursor.execute('PRAGMA table_info(books);')
cursor.fetchall()



[(0, 'id', 'INTEGER', 0, None, 1),
 (1, 'title', 'TEXT', 1, None, 0),
 (2, 'author', 'TEXT', 1, None, 0),
 (3, 'price', 'REAL', 1, None, 0),
 (4, 'stock_quantity', 'INTEGER', 1, None, 0)]

In [10]:
cursor.execute('PRAGMA table_info(customers);')
cursor.fetchall()



[(0, 'customer_id', 'INTEGER', 0, None, 1),
 (1, 'name', 'TEXT', 1, None, 0),
 (2, 'email', 'TEXT', 1, None, 0),
 (3, 'city', 'TEXT', 0, None, 0),
 (4, 'join_date', 'TEXT', 0, None, 0)]

In [11]:
cursor.execute('PRAGMA table_info(orders);')
cursor.fetchall()

[(0, 'order_id', 'INTEGER', 0, None, 1),
 (1, 'customer_id', 'INTEGER', 1, None, 0),
 (2, 'book_id', 'INTEGER', 1, None, 0),
 (3, 'quantity', 'INTEGER', 1, None, 0),
 (4, 'order_date', 'TEXT', 1, None, 0),
 (5, 'total_amount', 'REAL', 1, None, 0)]

In [12]:
books_data = [
    ('Python Programming', 'John Smith', 599.99, 25),
    ('Data Science Handbook', 'Jane Doe', 899.50, 15),
    ('Machine Learning Basics', 'Alan Turing', 1299.00, 10),
    ('SQL Essentials', 'Edgar Codd', 499.99, 30),
    ('Web Development', 'Tim Berners', 799.00, 20)
]

cursor.executemany('INSERT INTO books (title, author, price, stock_quantity) VALUES (?,?,?,?)', books_data)

In [13]:
customers_data = [
    ('Rahul Sharma', 'rahul@email.com', 'Mumbai', '2024-01-15'),
    ('Priya Patel', 'priya@email.com', 'Delhi', '2024-01-20'),
    ('Amit Kumar', 'amit@email.com', 'Bangalore', '2024-02-01'),
    ('Sneha Reddy', 'sneha@email.com', 'Hyderabad', '2024-02-10'),
    ('Vikram Singh', 'vikram@email.com', 'Mumbai', '2024-02-15')
]

cursor.executemany('INSERT INTO customers (name, email, city, join_date) VALUES (?,?,?,?)', customers_data)

In [14]:
orders_data = [
    (1, 1, 2, '2024-03-01', 1199.00),
    (1, 2, 1, '2024-03-02', 899.50),
    (2, 1, 1, '2024-03-03', 599.99),
    (2, 3, 1, '2024-03-05', 1299.00),
    (3, 4, 3, '2024-03-07', 1499.97),
    (4, 2, 1, '2024-03-10', 899.50),
    (5, 5, 2, '2024-03-12', 1598.00)
]

cursor.executemany('INSERT INTO orders (customer_id, book_id, quantity, order_date, total_amount) VALUES (?,?,?,?,?)', orders_data)

In [15]:
cursor.execute('SELECT * FROM books where price > 800 and stock_quantity > 10;')
for row in cursor.fetchall():
    print(row)


(2, 'Data Science Handbook', 'Jane Doe', 899.5, 15)


In [16]:
cursor.execute("SELECT * FROM customers where city = 'Mumbai';")
for row in cursor.fetchall():
    print(row)

(1, 'Rahul Sharma', 'rahul@email.com', 'Mumbai', '2024-01-15')
(5, 'Vikram Singh', 'vikram@email.com', 'Mumbai', '2024-02-15')


In [17]:
cursor.execute('SELECT count(*) FROM orders;')
for row in cursor.fetchall():
    print(row)

(7,)


In [18]:
cursor.execute('SELECT c.name, count(o.order_id) as total_orders FROM ORDERS o JOIN customers c on o.customer_id = c.customer_id group by o.customer_id order by total_orders DESC;')
for row in cursor.fetchall():
    print(row)

('Priya Patel', 2)
('Rahul Sharma', 2)
('Vikram Singh', 1)
('Sneha Reddy', 1)
('Amit Kumar', 1)


In [19]:
cursor.execute('SELECT sum(total_amount) FROM orders')
for row in cursor.fetchall():
    print(row)

(7994.96,)


In [20]:

import pandas as pd


books_df = pd.read_sql('SELECT * FROM books', conn)
books_df.head(3)

,id,title,author,price,stock_quantity
0,1,Python Programming,John Smith,599.99,25
1,2,Data Science Handbook,Jane Doe,899.50,15
2,3,Machine Learning Basics,Alan Turing,1299.00,10


In [21]:
customers_df = pd.read_sql('SELECT * FROM customers', conn)
customers_df.head(3)

,customer_id,name,email,city,join_date
0,1,Rahul Sharma,rahul@email.com,Mumbai,2024-01-15
1,2,Priya Patel,priya@email.com,Delhi,2024-01-20
2,3,Amit Kumar,amit@email.com,Bangalore,2024-02-01


In [22]:

orders_df = pd.read_sql('SELECT * FROM orders', conn)
orders_df.head(3)

,order_id,customer_id,book_id,quantity,order_date,total_amount
0,1,1,1,2,2024-03-01,1199.00
1,2,1,2,1,2024-03-02,899.50
2,3,2,1,1,2024-03-03,599.99


In [23]:
order_customers_df = orders_df.merge(customers_df, on='customer_id')
order_customers_df

,order_id,customer_id,book_id,quantity,order_date,total_amount,name,email,city,join_date
0,1,1,1,2,2024-03-01,1199.00,Rahul Sharma,rahul@email.com,Mumbai,2024-01-15
1,2,1,2,1,2024-03-02,899.50,Rahul Sharma,rahul@email.com,Mumbai,2024-01-15
2,3,2,1,1,2024-03-03,599.99,Priya Patel,priya@email.com,Delhi,2024-01-20
3,4,2,3,1,2024-03-05,1299.00,Priya Patel,priya@email.com,Delhi,2024-01-20
4,5,3,4,3,2024-03-07,1499.97,Amit Kumar,amit@email.com,Bangalore,2024-02-01
5,6,4,2,1,2024-03-10,899.50,Sneha Reddy,sneha@email.com,Hyderabad,2024-02-10
6,7,5,5,2,2024-03-12,1598.00,Vikram Singh,vikram@email.com,Mumbai,2024-02-15


In [24]:
full_merge_df = order_customers_df.merge(books_df, left_on='book_id', right_on='id')
full_merge_df

,order_id,customer_id,book_id,quantity,order_date,total_amount,name,email,city,join_date,id,title,author,price,stock_quantity
0,1,1,1,2,2024-03-01,1199.00,Rahul Sharma,rahul@email.com,Mumbai,2024-01-15,1,Python Programming,John Smith,599.99,25
1,2,1,2,1,2024-03-02,899.50,Rahul Sharma,rahul@email.com,Mumbai,2024-01-15,2,Data Science Handbook,Jane Doe,899.50,15
2,3,2,1,1,2024-03-03,599.99,Priya Patel,priya@email.com,Delhi,2024-01-20,1,Python Programming,John Smith,599.99,25
3,4,2,3,1,2024-03-05,1299.00,Priya Patel,priya@email.com,Delhi,2024-01-20,3,Machine Learning Basics,Alan Turing,1299.00,10
4,5,3,4,3,2024-03-07,1499.97,Amit Kumar,amit@email.com,Bangalore,2024-02-01,4,SQL Essentials,Edgar Codd,499.99,30
5,6,4,2,1,2024-03-10,899.50,Sneha Reddy,sneha@email.com,Hyderabad,2024-02-10,2,Data Science Handbook,Jane Doe,899.50,15
6,7,5,5,2,2024-03-12,1598.00,Vikram Singh,vikram@email.com,Mumbai,2024-02-15,5,Web Development,Tim Berners,799.00,20


In [53]:
raw_columns_df = full_merge_df[['order_id', 'name', 'city', 'title', 'quantity', 'total_amount']]
raw_columns_df.rename(columns={
    'order_id': 'Order ID',
    'name': 'Customer Name' ,
    'city': 'Customer City',
    'title': 'Book Title',
    'quantity' : 'Quantity' ,
    'total_amount' : 'Total Amount'
})


,Order ID,Customer Name,Customer City,Book Title,Quantity,Total Amount
0,1,Rahul Sharma,Mumbai,Python Programming,2,1199.00
1,2,Rahul Sharma,Mumbai,Data Science Handbook,1,899.50
2,3,Priya Patel,Delhi,Python Programming,1,599.99
3,4,Priya Patel,Delhi,Machine Learning Basics,1,1299.00
4,5,Amit Kumar,Bangalore,SQL Essentials,3,1499.97
5,6,Sneha Reddy,Hyderabad,Data Science Handbook,1,899.50
6,7,Vikram Singh,Mumbai,Web Development,2,1598.00


In [62]:
renamed_columns = raw_columns_df.rename(columns={
    'order_id': 'Order ID',
    'name': 'Customer Name' ,
    'city': 'Customer City',
    'title': 'Book Title',
    'quantity' : 'Quantity' ,
    'total_amount' : 'Total Amount'
})

renamed_columns



,Order ID,Customer Name,Customer City,Book Title,Quantity,Total Amount
0,1,Rahul Sharma,Mumbai,Python Programming,2,1199.00
1,2,Rahul Sharma,Mumbai,Data Science Handbook,1,899.50
2,3,Priya Patel,Delhi,Python Programming,1,599.99
3,4,Priya Patel,Delhi,Machine Learning Basics,1,1299.00
4,5,Amit Kumar,Bangalore,SQL Essentials,3,1499.97
5,6,Sneha Reddy,Hyderabad,Data Science Handbook,1,899.50
6,7,Vikram Singh,Mumbai,Web Development,2,1598.00


In [68]:
round(float(renamed_columns['Total Amount'].mean()), 2)

1142.14

In [83]:
renamed_columns['Customer City'].value_counts().reset_index()

,Customer City,count
0,Mumbai,3
1,Delhi,2
2,Bangalore,1
3,Hyderabad,1


In [85]:
renamed_columns.groupby('Book Title')['Quantity'].sum().idxmax()

'Python Programming'

In [92]:
discounts_data = {
    'book_id': [1, 2, 3, 4, 5],
    'discount_percent': [10, 15, 5, 20, 12]
}

discounts = pd.DataFrame(discounts_data)

discounts



,book_id,discount_percent
0,1,10
1,2,15
2,3,5
3,4,20
4,5,12


In [103]:
discounts.to_sql(name='discounts', con=conn, if_exists='replace', index=False)






5

In [106]:
cursor.execute("PRAGMA table_info(discounts);")
cursor.fetchall()

[(0, 'book_id', 'INTEGER', 0, None, 0),
 (1, 'discount_percent', 'INTEGER', 0, None, 0)]

In [113]:
query = '''SELECT b.title, b.price, d.discount_percent, ROUND(b.price * (1 - d.discount_percent / 100.0), 2) AS discounted_price  
    FROM books b 
    JOIN discounts d 
    ON b.id = d.book_id;'''



pd.read_sql(query, conn).rename(columns={
    'price': 'original_price',
    'discount_percent': 'discount%'
})

,title,original_price,discount%,discounted_price
0,Python Programming,599.99,10,539.99
1,Data Science Handbook,899.50,15,764.57
2,Machine Learning Basics,1299.00,5,1234.05
3,SQL Essentials,499.99,20,399.99
4,Web Development,799.00,12,703.12
